In [2]:
if (!requireNamespace("pacman", quietly = TRUE)){
    install.packages("pacman")
}
library(pacman)
pacman::p_load(tidyverse, lubridate, ggplot2, dplyr)


In [22]:
# Set seed for reproducibility
set.seed(45)
N <- 25

# Create the data frame
policy_data <- data.frame(
  Policy_ID = paste0("P", 1000:(1000 + N - 1)),
  Region = factor(sample(c("East", "West", "Central"), N, replace = TRUE, prob = c(0.4, 0.35, 0.25))),
  Exposure_in_Years = round(runif(N, min = 0.5, max = 5), 2),
  Num_Claims = rpois(N, lambda = 1.5),
  Premium_Amount = round(rnorm(N, mean = 1500, sd = 400), 2),
  Vehicle_Type = factor(sample(c("Sedan", "SUV", "Truck"), N, replace = TRUE, prob = c(0.5, 0.3, 0.2))),
  Inception_Date = seq(ymd('2022-01-01'), by = 'quarter', length.out = N),
  Agent_Rating = as.character(sample(c(rep("High", 18), rep("NA", 5), rep("Low", 2))))
)

In [23]:
# Introduce a few zero-exposure records for an edge case question
policy_data$Exposure_in_Years[c(10, 20)] <- 0
policy_data$Num_Claims[c(10, 20)] <- 0

# Display the initial structure
print("Data Structure:")
str(policy_data)
print("First 6 Rows:")
head(policy_data)


[1] "Data Structure:"
'data.frame':	25 obs. of  8 variables:
 $ Policy_ID        : chr  "P1000" "P1001" "P1002" "P1003" ...
 $ Region           : Factor w/ 3 levels "Central","East",..: 3 2 2 2 2 2 2 3 2 2 ...
 $ Exposure_in_Years: num  0.81 4.76 3.87 3.45 1.47 2.27 1.97 2.69 4.31 0 ...
 $ Num_Claims       : num  1 2 0 1 2 1 3 0 2 0 ...
 $ Premium_Amount   : num  2116 2373 1102 1379 1740 ...
 $ Vehicle_Type     : Factor w/ 3 levels "Sedan","SUV",..: 2 1 2 1 3 2 1 1 1 1 ...
 $ Inception_Date   : Date, format: "2022-01-01" "2022-04-01" ...
 $ Agent_Rating     : chr  "Low" "High" "High" "High" ...
[1] "First 6 Rows:"


,Policy_ID,Region,Exposure_in_Years,Num_Claims,Premium_Amount,Vehicle_Type,Inception_Date,Agent_Rating
,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<fct>,<date>,<chr>
1,P1000,West,0.81,1,2115.76,SUV,2022-01-01,Low
2,P1001,East,4.76,2,2373.20,Sedan,2022-04-01,High
3,P1002,East,3.87,0,1101.85,SUV,2022-07-01,High
4,P1003,East,3.45,1,1379.46,Sedan,2022-10-01,High
5,P1004,East,1.47,2,1740.20,Truck,2023-01-01,High
6,P1005,East,2.27,1,1354.68,SUV,2023-04-01,NA



QUESTION 1: Create a new variable called Frequency that shows claim frequency (number of claims/exposure). 

In [24]:
(policy_data_q1 <- policy_data %>%
        mutate(
            Frequency= ifelse(Exposure_in_Years == 0, 0, Num_Claims/Exposure_in_Years)
        )
)

Policy_ID,Region,Exposure_in_Years,Num_Claims,Premium_Amount,Vehicle_Type,Inception_Date,Agent_Rating,Frequency
<chr>,<fct>,<dbl>,<dbl>,<dbl>,<fct>,<date>,<chr>,<dbl>
P1000,West,0.81,1,2115.76,SUV,2022-01-01,Low,1.2345679
P1001,East,4.76,2,2373.20,Sedan,2022-04-01,High,0.4201681
P1002,East,3.87,0,1101.85,SUV,2022-07-01,High,0.0000000
P1003,East,3.45,1,1379.46,Sedan,2022-10-01,High,0.2898551
P1004,East,1.47,2,1740.20,Truck,2023-01-01,High,1.3605442
P1005,East,2.27,1,1354.68,SUV,2023-04-01,NA,0.4405286
P1006,East,1.97,3,1411.55,Sedan,2023-07-01,High,1.5228426
P1007,West,2.69,0,1420.76,Sedan,2023-10-01,High,0.0000000
P1008,East,4.31,2,1767.05,Sedan,2024-01-01,NA,0.4640371


QUESTION 2: Create a subset for policies in the East or West Region that have a premium greater than $1,800

In [25]:
(policy_data_q2 <- policy_data %>%
    filter(Region %in% c("East","West") & Premium_Amount > 1800))

Policy_ID,Region,Exposure_in_Years,Num_Claims,Premium_Amount,Vehicle_Type,Inception_Date,Agent_Rating
<chr>,<fct>,<dbl>,<dbl>,<dbl>,<fct>,<date>,<chr>
P1000,West,0.81,1,2115.76,SUV,2022-01-01,Low
P1001,East,4.76,2,2373.20,Sedan,2022-04-01,High
P1009,East,0.00,0,1847.01,Sedan,2024-04-01,NA
P1012,East,1.87,1,1867.55,Sedan,2025-01-01,Low
P1017,East,3.16,4,2121.72,Truck,2026-04-01,High


QUESTION 3: Calculate the following summary statistics by Vehicle Type: Total Exposure, Total Claims, and Total Number of Policies. 

In [26]:
(policy_data_q3 <- policy_data %>%
    group_by(Vehicle_Type) %>%
    summarize(
        Total_Exposure = sum(Exposure_in_Years),
        Total_Claims = sum(Num_Claims),
        Num_Policies = n()
    )
)


Vehicle_Type,Total_Exposure,Total_Claims,Num_Policies
<fct>,<dbl>,<dbl>,<int>
Sedan,33.08,15,12
SUV,23.11,15,9
Truck,6.66,8,4


QUESTION 4: Rearrange the dataset first by Region in decreasing (reverse alphabetical) order, and then by Exposure_in_Years in increasing order.

In [30]:
(policy_data_q4 <- policy_data %>%
    arrange(desc(Region), Exposure_in_Years)
)

Policy_ID,Region,Exposure_in_Years,Num_Claims,Premium_Amount,Vehicle_Type,Inception_Date,Agent_Rating
<chr>,<fct>,<dbl>,<dbl>,<dbl>,<fct>,<date>,<chr>
P1013,West,0.72,0,973.27,Sedan,2025-04-01,NA
P1000,West,0.81,1,2115.76,SUV,2022-01-01,Low
P1015,West,1.08,0,1121.80,SUV,2025-10-01,NA
P1023,West,1.99,4,1760.07,SUV,2027-10-01,High
P1014,West,2.00,0,1256.98,SUV,2025-07-01,High
P1007,West,2.69,0,1420.76,Sedan,2023-10-01,High
P1018,West,2.93,1,1632.26,SUV,2026-07-01,High
P1022,West,4.45,6,1786.32,SUV,2027-07-01,High
P1009,East,0.00,0,1847.01,Sedan,2024-04-01,NA
